# Diabetes — Intelex vs CuPy benchmark

Dataset: https://www.kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset

Models: LightGBM, XGBoost, Logistic Regression, Random Forest, AdaBoost, MLP.

Sections are grouped by implementation because `patch_sklearn()` patches the whole kernel: run **BareBones** and **CuPy** first, then the single patch cell, then **Intelex**.

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
import pandas as pd
import warnings
import cupy as cp
from sklearnex import patch_sklearn
from xgboost import DMatrix, train
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.neural_network import MLPClassifier


## BareBones (plain scikit-learn)

### LightGBM

In [ ]:
#Suppress FutureWarnings
warnings.filterwarnings("ignore", category=FutureWarning)
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    return data

def mainBareBones():
    # Load the dataset
    file_path = "diabetes_dataset.csv"  # Replace with the path to your CSV file
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning and Preprocessing
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    data.fillna(data.mean(numeric_only=True), inplace=True)  # Replace NaNs for numeric columns
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Encode categorical columns (gender, smoking_history)
    print("Encoding categorical columns...")
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes

    # Separate features (X) and target (y)
    print("\nSeparating features and target variable...")
    X = data.drop('diabetes', axis=1).values
    y = data['diabetes'].values
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split the data into training and testing sets
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Initialize the LightGBM Classifier
    print("\nInitializing the LightGBM classifier...")
    lgbm = LGBMClassifier(
        objective='binary', 
        random_state=42, 
        n_estimators=50
    )

    # Train the model
    print("\nTraining the LightGBM model...")
    lgbm.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = lgbm.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainBareBones()

### XGBoost

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    return data

def mainBareBones():
    # Load the dataset
    file_path = "diabetes_dataset.csv"  # Replace with the path to your CSV file
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning and Preprocessing
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    data.fillna(data.mean(numeric_only=True), inplace=True)  # Replace NaNs for numeric columns
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Encode categorical columns (gender, smoking_history)
    print("Encoding categorical columns...")
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes

    # Separate features (X) and target (y)
    print("\nSeparating features and target variable...")
    X = data.drop('diabetes', axis=1).values
    y = data['diabetes'].values
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split the data into training and testing sets
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    print("\nConverting data to DMatrix format for XGBoost...")
    dtrain = DMatrix(X_train, label=y_train)
    dtest = DMatrix(X_test, label=y_test)

    # Parameters for XGBoost
    params = {
        'objective': 'binary:logistic',
        'tree_method': 'hist',  # Optimized for GPU
        'device': 'cuda:0', # Use GPU,
        'random_state': 42 
    }

    # Train the model
    print("\nTraining the XGBoost model...")
    model = train(params, dtrain)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = (model.predict(dtest) > 0.5).astype(int)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy (Optimized): {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainBareBones()

### Logistic Regression

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns
    for col in data.select_dtypes(include='int'):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include='float'):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    # Convert categorical columns to category dtype
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes
    
    return data

def mainBareBones():
    # Load the dataset
    file_path = "diabetes_dataset.csv"  # Replace with your dataset's file path
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning and Preprocessing
    print("\nCleaning and preprocessing data...")
    print(f"Missing values before cleaning: {data.isnull().sum().sum()}")
    data.fillna(data.mean(numeric_only=True), inplace=True)  # Replace NaNs in numeric columns
    print(f"Missing values after cleaning: {data.isnull().sum().sum()}")

    # Optimize data types
    print("\nOptimizing data types...")
    data = optimize_datatypes(data)

    # Encode categorical variables (if applicable)
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes

    # Separate features (X) and target (y)
    print("\nSeparating features and target variable...")
    X = data.drop('diabetes', axis=1)
    y = data['diabetes']
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split the data into training and testing sets
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Scale features for better performance
    print("\nScaling features...")
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Initialize and train the logistic regression model
    print("\nTraining the logistic regression model...")
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainBareBones()

### Random Forest

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    return data

def mainBareBones():
    # Load the dataset
    file_path = "diabetes_dataset.csv"  # Replace with the path to your CSV file
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning and Preprocessing
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    data.fillna(data.mean(numeric_only=True), inplace=True)  # Handle missing values for numeric columns
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Encode categorical columns (gender, smoking_history)
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes
        print("Encoding categorical columns...")

    # Separate features (X) and target (y)
    print("\nSeparating features and target variable...")
    X = data.drop('diabetes', axis=1).values
    y = data['diabetes'].values
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split the data into training and testing sets
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Initialize and train the Random Forest Classifier
    print("\nTraining the Random Forest Classifier...")
    model = RandomForestClassifier(random_state=42, n_estimators=100)
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainBareBones()

### AdaBoost

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    return data

def mainBareBones():
    # Load the dataset
    file_path = "diabetes_dataset.csv"  # Replace with the path to your CSV file
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning and Preprocessing
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    data.fillna(data.mean(numeric_only=True), inplace=True)  # Replace NaNs for numeric columns
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Encode categorical columns (gender, smoking_history)
    print("Encoding categorical columns...")
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes

    # Separate features (X) and target (y)
    print("\nSeparating features and target variable...")
    X = data.drop('diabetes', axis=1).values
    y = data['diabetes'].values
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split the data into training and testing sets
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Train the model using AdaBoost
    print("\nTraining the AdaBoost model...")
    model = AdaBoostClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy (Optimized): {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainBareBones()


### MLP

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    return data

def mainBareBones():
    # Suppress warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    # Load the dataset
    file_path = "diabetes_dataset.csv"  # Replace with the path to your CSV file
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning and Preprocessing
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    data.fillna(data.mean(numeric_only=True), inplace=True)  # Replace NaNs for numeric columns
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Encode categorical columns (gender, smoking_history)
    print("Encoding categorical columns...")
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes

    # Separate features (X) and target (y)
    print("\nSeparating features and target variable...")
    X = data.drop('diabetes', axis=1).values
    y = data['diabetes'].values
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split the data into training and testing sets
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Train the model using MLPClassifier
    print("\nTraining the MLP model...")
    model = MLPClassifier(
        hidden_layer_sizes=(50, 30),  
        max_iter=1200,               
        learning_rate='adaptive',            
        early_stopping=True,        
        random_state=42              
    )
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainBareBones()


## CuPy

### LightGBM

In [ ]:
# Suppress FutureWarnings
warnings.filterwarnings("ignore", category=FutureWarning)

def optimize_datatypes(data):
    # Downcast numerical columns
    for col in data.select_dtypes(include=['int']):
        data[col] = cp.asnumpy(cp.array(data[col], dtype=cp.int32))
    for col in data.select_dtypes(include=['float']):
        data[col] = cp.asnumpy(cp.array(data[col], dtype=cp.float32))
    
    return data

def mainCuPY():
    # Load the dataset
    file_path = "diabetes_dataset.csv" 
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning and Preprocessing
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    data.fillna(data.mean(numeric_only=True), inplace=True)  # Replace NaNs for numeric columns
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes using CuPy
    print("\nOptimizing datatypes using CuPy...")
    data = optimize_datatypes(data)

    # Encode categorical columns (gender, smoking_history)
    print("Encoding categorical columns...")
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes

    # Separate features (X) and target (y)
    print("\nSeparating features and target variable...")
    X = cp.array(data.drop('diabetes', axis=1).values)
    y = cp.array(data['diabetes'].values)
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split the data into training and testing sets
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(cp.asnumpy(X), cp.asnumpy(y), test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Initialize the LightGBM Classifier
    print("\nInitializing the LightGBM classifier...")
    lgbm = LGBMClassifier(
        objective='binary', 
        random_state=42, 
        n_estimators=100
    )

    # Train the model
    print("\nTraining the LightGBM model...")
    lgbm.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = lgbm.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainCuPY()


### XGBoost

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    return data

def mainCuPY():
    # Load the dataset
    file_path = "diabetes_dataset.csv"  # Replace with the path to your CSV file
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning and Preprocessing
    print("\nCleaning and preprocessing data...")
    print(f"Missing values before cleaning: {data.isnull().sum().sum()}")
    data.fillna(data.mean(numeric_only=True), inplace=True)  # Replace NaNs with column means
    print(f"Missing values after cleaning: {data.isnull().sum().sum()}")

    # Optimize datatypes
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Encode categorical columns (gender, smoking_history)
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes

    # Separate features (X) and target (y)
    print("\nSeparating features and target variable...")
    X = data.drop('diabetes', axis=1).values
    y = data['diabetes'].values
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split the data into training and testing sets
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Convert to CuPy arrays for GPU acceleration
    print("\nTransferring data to GPU...")
    X_train_cp = cp.asarray(X_train, dtype=cp.float32)
    X_test_cp = cp.asarray(X_test, dtype=cp.float32)
    y_train_cp = cp.asarray(y_train, dtype=cp.int32)
    y_test_cp = cp.asarray(y_test, dtype=cp.int32)

    print("\nConverting data to DMatrix format for XGBoost...")
    dtrain = DMatrix(X_train_cp, label=y_train_cp)
    dtest = DMatrix(X_test_cp, label=y_test_cp)

    # Parameters for XGBoost
    params = {
        'objective': 'binary:logistic',
        'tree_method': 'hist',  # Optimized for GPU
        'device': 'cuda:0',                # Use GPU device 0
        'random_state': 42
    }

    # Train the model
    print("\nTraining the XGBoost model...")
    model = train(params, dtrain)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = (cp.asarray(model.predict(dtest)) > 0.5).astype(int)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(cp.asnumpy(y_test_cp), cp.asnumpy(y_pred))
    print(f"Accuracy (CuPy Optimized): {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(cp.asnumpy(y_test_cp), cp.asnumpy(y_pred)))

if __name__ == "__main__":
    mainCuPY()


### Logistic Regression

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    return data

def mainCuPY():
    # Load the dataset
    file_path = "diabetes_dataset.csv"  # Replace with your file path
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning and Preprocessing
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    data.fillna(data.mean(numeric_only=True), inplace=True)  # Fill NaNs for numeric columns
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes
    print("\nOptimizing data types...")
    data = optimize_datatypes(data)

    # Encode categorical columns (gender, smoking_history) if necessary
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes

    # Separate features (X) and target (y)
    print("\nSeparating features and target variable...")
    X = data.drop('diabetes', axis=1).values
    y = data['diabetes'].values
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split the data into training and testing sets
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Convert to CuPy arrays for GPU acceleration
    print("\nTransferring data to GPU using CuPy...")
    X_train_cp = cp.asarray(X_train, dtype=cp.float32)
    X_test_cp = cp.asarray(X_test, dtype=cp.float32)
    y_train_cp = cp.asarray(y_train, dtype=cp.int32)
    y_test_cp = cp.asarray(y_test, dtype=cp.int32)

    # Standardize features
    print("\nScaling features...")
    scaler = StandardScaler()
    X_train_scaled = cp.asarray(scaler.fit_transform(cp.asnumpy(X_train_cp)))
    X_test_scaled = cp.asarray(scaler.transform(cp.asnumpy(X_test_cp)))

    # Train the logistic regression model
    print("\nTraining the logistic regression model...")
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(cp.asnumpy(X_train_scaled), cp.asnumpy(y_train_cp))
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(cp.asnumpy(X_test_scaled))

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(cp.asnumpy(y_test_cp), y_pred)
    print(f"Accuracy (CuPy Optimized): {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(cp.asnumpy(y_test_cp), y_pred))

if __name__ == "__main__":
    mainCuPY()

### Random Forest

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    return data

def mainCuPY():
    # Load the dataset
    file_path = "diabetes_dataset.csv"  # Replace with the path to your CSV file
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning and Preprocessing
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    data.fillna(data.mean(numeric_only=True), inplace=True)  # Handle only numeric columns
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes to reduce memory usage
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Encode categorical columns (gender, smoking_history) if necessary
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes
        print("Encoding categorical columns...")

    # Separate features (X) and target (y)
    print("\nSeparating features and target variable...")
    X = data.drop('diabetes', axis=1).values
    y = data['diabetes'].values
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split the data into training and testing sets
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Convert data to CuPy arrays for GPU acceleration
    print("\nTransferring data to GPU using CuPy...")
    X_train_cp = cp.asarray(X_train, dtype=cp.float32)  # Specify dtype for efficiency
    X_test_cp = cp.asarray(X_test, dtype=cp.float32)
    y_train_cp = cp.asarray(y_train, dtype=cp.int32)
    y_test_cp = cp.asarray(y_test, dtype=cp.int32)

    # Initialize and train the Random Forest Classifier
    print("\nTraining the Random Forest Classifier...")
    model = RandomForestClassifier(random_state=42, n_estimators=100)
    model.fit(cp.asnumpy(X_train_cp), cp.asnumpy(y_train_cp))  # Use NumPy arrays for training
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(cp.asnumpy(X_test_cp))

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(cp.asnumpy(y_test_cp), y_pred)
    print(f"Accuracy (CuPy Optimized): {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(cp.asnumpy(y_test_cp), y_pred))

if __name__ == "__main__":
    mainCuPY()


### AdaBoost

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns
    for col in data.select_dtypes(include=['int']):
        cupy_array = cp.array(data[col], dtype=cp.int32)  # Convert to CuPy array
        data[col] = cupy_array.get()  # Convert back to NumPy array
    for col in data.select_dtypes(include=['float']):
        cupy_array = cp.array(data[col], dtype=cp.float32)  # Convert to CuPy array
        data[col] = cupy_array.get()  # Convert back to NumPy array
    
    return data

def mainCuPY():
    # Load the dataset
    file_path = "diabetes_dataset.csv"  # Replace with the path to your CSV file
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning and Preprocessing
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    data.fillna(data.mean(numeric_only=True), inplace=True)  # Replace NaNs for numeric columns
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Encode categorical columns (gender, smoking_history)
    print("Encoding categorical columns...")
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes

    # Separate features (X) and target (y)
    print("\nSeparating features and target variable...")
    X = cp.array(data.drop('diabetes', axis=1).values)  # Convert to CuPy array
    y = cp.array(data['diabetes'].values)  # Convert to CuPy array
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split the data into training and testing sets
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(cp.asnumpy(X), cp.asnumpy(y), test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Train the model using AdaBoost
    print("\nTraining the AdaBoost model...")
    model = AdaBoostClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy (Optimized): {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainCuPY()


### MLP

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns using CuPy for GPU acceleration
    for col in data.select_dtypes(include=['int']):
        data[col] = cp.array(data[col], dtype=cp.int32).get()
    for col in data.select_dtypes(include=['float']):
        data[col] = cp.array(data[col], dtype=cp.float32).get()
    
    return data

def mainCuPY():
    # Suppress warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    # Load the dataset
    file_path = "diabetes_dataset.csv"  # Replace with the path to your CSV file
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning and Preprocessing
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    data.fillna(data.mean(numeric_only=True), inplace=True)  # Replace NaNs for numeric columns
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Encode categorical columns (gender, smoking_history)
    print("Encoding categorical columns...")
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes

    # Separate features (X) and target (y)
    print("\nSeparating features and target variable...")
    X = cp.array(data.drop('diabetes', axis=1).values, dtype=cp.float32).get()
    y = cp.array(data['diabetes'].values, dtype=cp.float32).get()
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split the data into training and testing sets
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Train the model using MLPClassifier
    print("\nTraining the MLP model...")
    model = MLPClassifier(
        hidden_layer_sizes=(50, 30),  
        max_iter=1200,               
        learning_rate='adaptive',            
        early_stopping=True,        
        random_state=42              
    )
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainCuPY()


## Intelex (scikit-learn-intelex)

Apply the Intel patch once; everything below runs patched. The patch cell re-imports the scikit-learn classes: `patch_sklearn()` only replaces them inside the `sklearn` modules, so names imported in the first cell would otherwise still point at stock scikit-learn.

In [ ]:
patch_sklearn()  # Apply Intel optimizations

# Re-import after patching: names imported earlier still point at the unpatched classes.
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.neural_network import MLPClassifier

### LightGBM

In [ ]:
#Suppress FutureWarnings
warnings.filterwarnings("ignore", category=FutureWarning)
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    return data

def mainBareBones():
    # Load the dataset
    file_path = "diabetes_dataset.csv"  # Replace with the path to your CSV file
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning and Preprocessing
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    data.fillna(data.mean(numeric_only=True), inplace=True)  # Replace NaNs for numeric columns
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Encode categorical columns (gender, smoking_history)
    print("Encoding categorical columns...")
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes

    # Separate features (X) and target (y)
    print("\nSeparating features and target variable...")
    X = data.drop('diabetes', axis=1).values
    y = data['diabetes'].values
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split the data into training and testing sets
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Initialize the LightGBM Classifier
    print("\nInitializing the LightGBM classifier...")
    lgbm = LGBMClassifier(
        objective='binary', 
        random_state=42, 
        n_estimators=50
    )

    # Train the model
    print("\nTraining the LightGBM model...")
    lgbm.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = lgbm.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainBareBones()

### XGBoost

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    return data

def mainIntelex():
    # Load the dataset
    file_path = "diabetes_dataset.csv"  # Replace with the path to your CSV file
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning and Preprocessing
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    data.fillna(data.mean(numeric_only=True), inplace=True)  # Replace NaNs for numeric columns
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Encode categorical columns (gender, smoking_history)
    print("Encoding categorical columns...")
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes

    # Separate features (X) and target (y)
    print("\nSeparating features and target variable...")
    X = data.drop('diabetes', axis=1).values
    y = data['diabetes'].values
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split the data into training and testing sets
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    print("\nConverting data to DMatrix format for XGBoost...")
    dtrain = DMatrix(X_train, label=y_train)
    dtest = DMatrix(X_test, label=y_test)

    # Parameters for XGBoost
    params = {
        'objective': 'binary:logistic',
        'tree_method': 'hist',  # Optimized for GPU
        'device': 'cuda:0', # Use GPU,
        'random_state': 42 
    }

    # Train the model
    print("\nTraining the XGBoost model...")
    model = train(params, dtrain)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = (model.predict(dtest) > 0.5).astype(int)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy (Optimized): {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainIntelex()

### Logistic Regression

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns
    for col in data.select_dtypes(include='int'):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include='float'):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    # Convert categorical columns to category dtype
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes
    
    return data

def mainIntelex():
    # Load the dataset
    file_path = "diabetes_dataset.csv"  # Replace with your dataset's file path
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning and Preprocessing
    print("\nCleaning and preprocessing data...")
    print(f"Missing values before cleaning: {data.isnull().sum().sum()}")
    data.fillna(data.mean(numeric_only=True), inplace=True)  # Replace NaNs in numeric columns
    print(f"Missing values after cleaning: {data.isnull().sum().sum()}")

    # Optimize data types
    print("\nOptimizing data types...")
    data = optimize_datatypes(data)

    # Encode categorical variables (if applicable)
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes

    # Separate features (X) and target (y)
    print("\nSeparating features and target variable...")
    X = data.drop('diabetes', axis=1)
    y = data['diabetes']
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split the data into training and testing sets
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Scale features for better performance
    print("\nScaling features...")
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Initialize and train the logistic regression model
    print("\nTraining the logistic regression model...")
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainIntelex()

### Random Forest

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    return data

def mainIntelex():
    # Load the dataset
    file_path = "diabetes_dataset.csv"  # Replace with the path to your CSV file
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning and Preprocessing
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    data.fillna(data.mean(numeric_only=True), inplace=True)  # Handle missing values for numeric columns
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Encode categorical columns (gender, smoking_history)
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes
        print("Encoding categorical columns...")

    # Separate features (X) and target (y)
    print("\nSeparating features and target variable...")
    X = data.drop('diabetes', axis=1).values
    y = data['diabetes'].values
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split the data into training and testing sets
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Initialize and train the Random Forest Classifier
    print("\nTraining the Random Forest Classifier...")
    model = RandomForestClassifier(random_state=42, n_estimators=100)
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainIntelex()

### AdaBoost

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    return data

def mainIntelex():
    # Load the dataset
    file_path = "diabetes_dataset.csv"  # Replace with the path to your CSV file
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning and Preprocessing
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    data.fillna(data.mean(numeric_only=True), inplace=True)  # Replace NaNs for numeric columns
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Encode categorical columns (gender, smoking_history)
    print("Encoding categorical columns...")
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes

    # Separate features (X) and target (y)
    print("\nSeparating features and target variable...")
    X = data.drop('diabetes', axis=1).values
    y = data['diabetes'].values
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split the data into training and testing sets
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Train the model using AdaBoost
    print("\nTraining the AdaBoost model...")
    model = AdaBoostClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy (Optimized): {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainIntelex()


### MLP

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    
    return data

def mainIntelex():
    # Suppress warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    # Load the dataset
    file_path = "diabetes_dataset.csv"  # Replace with the path to your CSV file
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning and Preprocessing
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    data.fillna(data.mean(numeric_only=True), inplace=True)  # Replace NaNs for numeric columns
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Encode categorical columns (gender, smoking_history)
    print("Encoding categorical columns...")
    categorical_cols = ['gender', 'smoking_history']
    for col in categorical_cols:
        if col in data.columns:
            data[col] = data[col].astype('category').cat.codes

    # Separate features (X) and target (y)
    print("\nSeparating features and target variable...")
    X = data.drop('diabetes', axis=1).values
    y = data['diabetes'].values
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split the data into training and testing sets
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Train the model using MLPClassifier
    print("\nTraining the MLP model...")
    model = MLPClassifier(
        hidden_layer_sizes=(50, 30),  
        max_iter=1200,               
        learning_rate='adaptive',            
        early_stopping=True,        
        random_state=42              
    )
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainIntelex()
